<a href="https://colab.research.google.com/github/Jalilnkh/PyTorch-with-Examples-2024/blob/parts/en_azb_mt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [84]:
#Checking if GPU is running or not
!nvidia-smi

Thu Dec 12 20:31:33 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   70C    P0              29W /  70W |   4213MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [85]:
!pip install datasets transformers[sentencepiece] sacrebleu -q

In [86]:
import os
import sys
import transformers
import tensorflow as tf
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import TFAutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from transformers import AdamWeightDecay
from transformers import AutoTokenizer, TFAutoModelForSeq2SeqLM

In [87]:
model_checkpoint = "Helsinki-NLP/opus-mt-en-az"

In [88]:
raw_datasets = load_dataset("Kartal-Ol/en-azb-548k")

In [89]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 548900
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 4000
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 4000
    })
})

In [90]:
raw_datasets['train'][1]

{'translation': {'azb': '( سیزین اؤزونوزه سلام اوْلماسین ! )',
  'en': 'No welcome for you !'}}

In [91]:
raw_datasets['train'][100000]

{'translation': {'azb': 'یئهووا هابیلین قُربانینی نه\u200cیه گؤره تقدیر ائتدی ؟',
  'en': 'Why did Jehovah look with favor on Abel ’ s offering ?'}}

In [92]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [93]:
tokenizer("Hello, this is a sentence!")

{'input_ids': [1236, 436, 6, 106, 18, 16, 9415, 186, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [94]:
tokenizer(["Hello, this is a sentence!", "This is another sentence."])

{'input_ids': [[1236, 436, 6, 106, 18, 16, 9415, 186, 0], [288, 18, 455, 9415, 5, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]]}

In [95]:
with tokenizer.as_target_tokenizer():
    print(tokenizer(['( سیزین اؤزونوزه سلام اوْلماسین !']))

{'input_ids': [[7, 28, 1, 23200, 28, 1, 23200, 1, 28, 1, 28, 1, 23200, 14, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}


In [96]:
max_input_length = 128
max_target_length = 128

source_lang = "en"
target_lang = "azb"


def preprocess_function(examples):
    inputs = [ex[source_lang] for ex in examples["translation"]]
    targets = [ex[target_lang] for ex in examples["translation"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    # Setup the tokenizer for targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [97]:
preprocess_function(raw_datasets["train"][:2])

{'input_ids': [[10946, 68, 1010, 328, 54, 8264, 328, 12883, 37, 2740, 257, 21234, 20344, 56, 813, 2, 12883, 37, 2740, 257, 21234, 970, 1295, 56, 187, 301, 29, 2856, 14, 800, 90, 97, 221, 16, 7344, 11, 2223, 155, 62, 3277, 3, 0], [563, 9068, 30, 17, 14, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1]], 'labels': [[28, 1, 23199, 1, 23200, 28, 1, 28, 1, 23199, 1, 23200, 1, 23200, 7, 28, 1, 28, 1, 28, 1, 28, 1, 23199, 1, 8, 28, 1, 23200, 1, 28, 1, 28, 1, 26, 28, 89, 28, 1, 28, 1, 28, 1, 28, 23199, 1, 23200, 28, 1, 23200, 1, 23200, 28, 1, 23200, 1, 23200, 28, 1, 23200, 1, 23200, 28, 1, 23199, 28, 1, 28, 1, 3, 0], [7, 28, 1, 23200, 28, 1, 23200, 1, 28, 1, 28, 1, 23200, 14, 8, 0]]}

In [98]:
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

In [99]:
model = TFAutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

All model checkpoint layers were used when initializing TFMarianMTModel.

All the layers of TFMarianMTModel were initialized from the model checkpoint at Helsinki-NLP/opus-mt-en-az.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFMarianMTModel for predictions without further training.


In [106]:
batch_size = 64
learning_rate = 2e-5
weight_decay = 0.01
num_train_epochs = 2

In [101]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, return_tensors="tf")

In [102]:
generation_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, return_tensors="tf", pad_to_multiple_of=128)

In [103]:
train_dataset = model.prepare_tf_dataset(
    tokenized_datasets["train"],
    batch_size=batch_size,
    shuffle=True,
    collate_fn=data_collator,
)

In [104]:
validation_dataset = model.prepare_tf_dataset(
    tokenized_datasets["validation"],
    batch_size=batch_size,
    shuffle=False,
    collate_fn=data_collator,
)

In [105]:
generation_dataset = model.prepare_tf_dataset(
    tokenized_datasets["validation"],
    batch_size=8,
    shuffle=False,
    collate_fn=generation_data_collator,
)

In [107]:
optimizer = AdamWeightDecay(learning_rate=learning_rate, weight_decay_rate=weight_decay)
model.compile(optimizer=optimizer)

In [ ]:
model.fit(train_dataset, validation_data=validation_dataset, epochs=1)
model.save_pretrained("tf_model/")

3570/8576 [===========>..................] - ETA: 1:21:33 - loss: 1.0466

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = TFAutoModelForSeq2SeqLM.from_pretrained("tf_model/")

input_text  = "My name Bappy. My youtube channel name is DSwithBappy"

tokenized = tokenizer([input_text], return_tensors='np')
out = model.generate(**tokenized, max_length=128)
print(out)

In [ ]:
with tokenizer.as_target_tokenizer():
    print(tokenizer.decode(out[0], skip_special_tokens=True))